# 06 — Refinery events and capacity

Validate the pre-specified event registry and published nominal-capacity checkpoints. Do not interpolate capacity as if it were observed production.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)


In [ ]:
events = pd.read_csv(PATHS.reference / "refinery_events.csv")
capacity = pd.read_csv(PATHS.reference / "refinery_capacity_points.csv")
assert events["source_url"].notna().all()
assert capacity["source_url"].notna().all()
display(events)
display(capacity)


In [ ]:
regimes = pd.DataFrame({"year": range(2005, 2025)})
regimes["refining_regime"] = np.select(
    [regimes["year"] <= 2020, regimes["year"] == 2021, regimes["year"] >= 2022],
    ["two_refineries", "transition", "sines_only"],
    default="unknown",
)
regimes["n_operating_refineries_regime"] = np.select(
    [regimes["year"] <= 2020, regimes["year"] >= 2022], [2, 1], default=np.nan
)
persist_dataframe(regimes, PATHS.processed / "refining_regime_annual.csv", key_columns=["year"])
regimes
